# Transformer 编码器–解码器：逐块运行

这个笔记本把 [transformer_encoder_decoder.py](./transformer_encoder_decoder.py) 拆成一个个小代码块，按书中第 2 章 *Transformer: From Input to Output* 的 11 步顺序，从 token id 一路走到 loss 和生成。

每一步都分三段：

1. **数学**：这一步的公式，以及输入和输出的形状。
2. **代码**：和 `.py` 里一字不差的类或函数。
3. **运行**：在同一个例子上跑一遍，打印形状，并用 `assert` 核对代码算的就是公式。

和 `.py` 相同的约定：

- **Post-norm**，即论文里的 $x = \operatorname{LN}(x + \operatorname{Sublayer}(x))$。
- **不用 dropout，也不共享权重**，这样每一行代码对应一个公式。
- **mask 是布尔张量**，`True` 表示"这个 key 可以被读"。

**怎么运行。** 在 VS Code 里选 `.venv-transformer` 作为 kernel（torch 2.8，CPU 就够），从上到下依次运行。`.py` 文件保持不变：测试、`lora_linear.py` 和 `vlm_vit_projector_llm.py` 仍然从它 import。

| 部分 | 内容 |
|---|---|
| 0 | 导入与例子 |
| 1 | 第 1 步：embedding + 位置编码；思考题：绝对位置还是相对位置 |
| 2 | 第 2 步：多头自注意力 |
| 3–4 | 第 3–4 步：Add & Norm、FFN，组装 `EncoderLayer` |
| 5 | 第 5 步：解码器输入，以及两种 mask |
| 6–8 | 第 6–8 步：掩码自注意力、交叉注意力、FFN，组装 `DecoderLayer` |
| 9 | 第 9 步：logits，组装整个 `Transformer` |
| 10 | 第 10 步：loss 与一次参数更新 |
| 11 | 第 11 步：贪心生成 |
| RoPE | 旋转位置编码：第 1 步的现代替代方案 |
| main | 一次跑完 `.py` 的 `main()` |

## 0. 导入与例子

### 0.1 导入

`torch.nn` 提供带参数的模块，例如 `nn.Embedding`、`nn.Linear`、`nn.LayerNorm`。`torch.nn.functional`（简写 `F`）提供没有参数的函数，例如 `F.softmax`、`F.relu`、`F.cross_entropy`。整个模型只用这三种模块搭成，不用 `nn.Transformer` 或 `nn.MultiheadAttention`。

In [ ]:
import math

import torch
from torch import Tensor, nn
from torch.nn import functional as F

torch.manual_seed(0)
print("torch", torch.__version__)

torch 2.8.0


### 0.2 例子：一句话贯穿全程

源句 `we watch a short video`，目标句 `nous regardons video`。每种语言一个小词表，`PAD = 0`，`BOS = 1`，`EOS = 2`。

$$
\begin{aligned}
\text{source ids}\quad \mathbf{s} &= (3,\ 4,\ 5,\ 6,\ 7) && \text{we watch a short video},\ n = 5 \\
\text{decoder input}\quad \mathbf{t} &= (1,\ 3,\ 4,\ 5) && \text{BOS nous regardons video},\ m = 4 \\
\text{labels}\quad \mathbf{y} &= (3,\ 4,\ 5,\ 2) && \text{nous regardons video EOS},\ y_j = t_{j+1}
\end{aligned}
$$

- **id 只是行号。** `we` 是 3，只因为它在源词表里的下标是 3（从 0 数，前面是 `<pad>`、`<bos>`、`<eos>`）。目标词表是另一张表，目标 id 3（`nous`）和源 id 3 没有任何关系。
- **解码器输入是目标句右移一位。** 开头放 `BOS`，每个位置的标签是它右边那个 token。

| 位置 $j$ | 输入 $t_j$ | 能读到的 token（第 6 步的 mask） | 标签 $y_j$ |
|---|---|---|---|
| 0 | BOS | BOS | nous |
| 1 | nous | BOS, nous | regardons |
| 2 | regardons | BOS, nous, regardons | video |
| 3 | video | BOS, nous, regardons, video | EOS |

**下标从 0 数，和代码一致**：$t_0$ 是 BOS，位置 $j$ 的标签是 $y_j = t_{j+1}$。书里的公式从 1 数，说的是同一件事。

每个位置的答案都在它右边一格，而 mask 遮住右边的一切，所以没有位置能看到自己的答案。把整个已知目标一次性喂进去叫 teacher forcing：一次前向传播就训练全部 $m$ 个预测。

尺寸：模型宽度 $d = 8$，头数 $h = 2$，每头宽度 $d_k = d/h = 4$，FFN 宽度 $d_{ff} = 32$，词表 $V_{\text{src}} = 8$、$V_{\text{tgt}} = 10$。PyTorch 在每个形状前面多一个 batch 维 $B$，这里 $B = 1$。

In [2]:
src_vocab = ["<pad>", "<bos>", "<eos>", "we", "watch", "a", "short", "video"]
tgt_vocab = ["<pad>", "<bos>", "<eos>", "nous", "regardons", "video", "une", "courte", "tu", "regardes"]
PAD, BOS, EOS = 0, 1, 2
d_model, num_heads, d_ff = 8, 2, 32

s = torch.tensor([[3, 4, 5, 6, 7]])   # we watch a short video     [B=1, n=5]
t = torch.tensor([[1, 3, 4, 5]])      # BOS nous regardons video   [B=1, m=4]
y = torch.tensor([[3, 4, 5, 2]])      # nous regardons video EOS   [B=1, m=4]


def words(ids, vocab):
    return " ".join(vocab[i] for i in ids.tolist())


print("s =", s[0].tolist(), "=", words(s[0], src_vocab))
print("t =", t[0].tolist(), "=", words(t[0], tgt_vocab))
print("y =", y[0].tolist(), "=", words(y[0], tgt_vocab))
assert torch.equal(t[0, 1:], y[0, :-1])   # y_j = t_{j+1}: the labels are the decoder input shifted left

s = [3, 4, 5, 6, 7] = we watch a short video
t = [1, 3, 4, 5] = <bos> nous regardons video
y = [3, 4, 5, 2] = nous regardons video <eos>


## 1. 第 1 步：token id → embedding + 位置

**输入** $\mathbf{s} \in \{0,\dots,V_{\text{src}}-1\}^{n}$，**输出** $X \in \mathbb{R}^{n \times d}$。

$$
X = \sqrt{d}\; E_{\text{src}}[\mathbf{s}] + P_{0:n-1}, \qquad E_{\text{src}} \in \mathbb{R}^{V_{\text{src}} \times d}\ \text{(learned)}, \qquad P \in \mathbb{R}^{L_{\max} \times d}\ \text{(fixed)}
$$

两部分各管一件事：$E_{\text{src}}[\mathbf{s}]$ 说明"这是哪个词"，$P_{0:n-1}$（$P$ 的前 $n$ 行）说明"它在第几个位置"。没有 $P$ 的话，自注意力分不清顺序，`we watch` 和 `watch we` 看起来一样（第 2 步末尾会验证这一点）。下面先写 $P$，再写 embedding。

### 1a. 正弦位置编码 `sinusoidal_positions`

$$
P_{p,\,2i} = \sin\!\left(\frac{p}{10000^{2i/d}}\right), \qquad P_{p,\,2i+1} = \cos\!\left(\frac{p}{10000^{2i/d}}\right), \qquad i = 0,\dots,\tfrac{d}{2}-1
$$

**输入** `max_len` $= L$ 和 `d_model` $= d$，**输出** $P$，形状 `[L, d]`。

每一对特征 $(2i,\ 2i+1)$ 是同一个角度的 sin 和 cos，这个角度随位置匀速增长，速度是 $\omega_i = 10000^{-2i/d}$。第 0 对每走一个位置转 1 弧度，后面的对越来越慢。先把函数体逐行拆开看：

In [3]:
L, d = 5, d_model
position = torch.arange(L, dtype=torch.float32)[:, None]   # [L, 1]: 0, 1, 2, 3, 4 as a column
two_i = torch.arange(0, d, 2, dtype=torch.float32)         # [d/2]: 0, 2, 4, 6
angle = position / (10000 ** (two_i / d))                  # [L, 1] / [d/2] broadcasts to [L, d/2]
print("position", list(position.shape), "  two_i", two_i.tolist(), "  angle", list(angle.shape))
print("omega_i = 1 / 10000^(2i/d):", ", ".join(f"{w:g}" for w in (1 / 10000 ** (two_i / d)).tolist()))
print(angle)

position [5, 1]   two_i [0.0, 2.0, 4.0, 6.0]   angle [5, 4]
omega_i = 1 / 10000^(2i/d): 1, 0.1, 0.01, 0.001
tensor([[    0.00,     0.00,     0.00,     0.00],
        [    1.00,     0.10,     0.01,     0.00],
        [    2.00,     0.20,     0.02,     0.00],
        [    3.00,     0.30,     0.03,     0.00],
        [    4.00,     0.40,     0.04,     0.00]])


`angle[p, i]` 就是 $p\,\omega_i$。偶数列放 sin，奇数列放 cos，就得到完整的函数：

In [4]:
def sinusoidal_positions(max_len: int, d_model: int) -> Tensor:
    """P[p, 2i] = sin(p / 10000^(2i/d)),  P[p, 2i+1] = cos(p / 10000^(2i/d))."""
    assert d_model % 2 == 0, "sin/cos pairs need an even d_model"
    position = torch.arange(max_len, dtype=torch.float32)[:, None]      # [L, 1]
    two_i = torch.arange(0, d_model, 2, dtype=torch.float32)            # [d/2]: 0, 2, 4, ...
    angle = position / (10000 ** (two_i / d_model))                     # [L, d/2]
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe                                                           # [L, d]

运行一下，核对几个具体的值：

- **位置 0** 的行是 $(0, 1, 0, 1, \dots)$，因为 $\sin 0 = 0$，$\cos 0 = 1$。
- **位置 1** 的前两个值是 $\sin 1 = 0.841$ 和 $\cos 1 = 0.540$。
- **每一行的范数**都是 $\sqrt{d/2} = 2$，因为每一对贡献 $\sin^2 + \cos^2 = 1$。
- **两个位置的点积只取决于距离**：$P_p \cdot P_q = \sum_i \cos\big((p - q)\,\omega_i\big)$。

In [5]:
P = sinusoidal_positions(5, d_model)
print(P)
assert torch.allclose(P[1, :2], torch.tensor([math.sin(1), math.cos(1)]))
print("row norms:", P.norm(dim=1))                                   # all sqrt(d/2) = 2
print(f"P[1]·P[3] = {(P[1] @ P[3]).item():.4f}   P[2]·P[4] = {(P[2] @ P[4]).item():.4f}   (distance 2 both times)")

tensor([[     0.00,      1.00,      0.00,      1.00,      0.00,      1.00,
              0.00,      1.00],
        [     0.84,      0.54,      0.10,      1.00,      0.01,      1.00,
              0.00,      1.00],
        [     0.91,     -0.42,      0.20,      0.98,      0.02,      1.00,
              0.00,      1.00],
        [     0.14,     -0.99,      0.30,      0.96,      0.03,      1.00,
              0.00,      1.00],
        [    -0.76,     -0.65,      0.39,      0.92,      0.04,      1.00,
              0.00,      1.00]])
row norms: tensor([2., 2., 2., 2., 2.])
P[1]·P[3] = 2.5637   P[2]·P[4] = 2.5637   (distance 2 both times)


### 1b. `TokenEmbedding`：查表、缩放、加位置

- **`nn.Embedding(V, d)` 就是表 $E \in \mathbb{R}^{V \times d}$。** `E[ids]` 按行号取出若干行，是查表，不是用 id 去乘矩阵。
- **`padding_idx=pad_id`** 把 PAD 那一行初始化为 0，训练时也不给它梯度。
- **`register_buffer`** 把固定的 $P$ 存进模型。它会跟着 `model.to(device)` 一起移动，也会存进 `state_dict`，但它不是参数，优化器不会更新它。
- **乘 $\sqrt{d}$ 是论文的约定。** 论文让 embedding 和输出投影共用同一个矩阵，所以 embedding 按输出层的尺度初始化得很小（每个元素约 $1/\sqrt{d}$）。乘上 $\sqrt{d}$ 之后，每个元素约为 1，每行的范数约为 $\sqrt{d}$，和 $P$（元素在 $[-1, 1]$ 之间，每行范数 $\sqrt{d/2}$）同一量级。PyTorch 默认按 $\mathcal{N}(0, 1)$ 初始化，这个因子就不是必需的（GPT-2、LLaMA 不乘，Gemma 保留）。这里保留它，只是为了和公式一致。

**输入** ids `[B, n]`，**输出** `[B, n, d]`。`forward` 只取 $P$ 的前 $n$ 行。

In [6]:
class TokenEmbedding(nn.Module):
    """Steps 1 and 5:  X = sqrt(d) * E[ids] + P[:n]."""

    def __init__(self, vocab_size: int, d_model: int, pad_id: int, max_len: int = 512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)  # E: [V, d], learned
        self.scale = math.sqrt(d_model)
        self.register_buffer("positions", sinusoidal_positions(max_len, d_model))  # P: [L, d], fixed

    def forward(self, ids: Tensor) -> Tensor:
        n = ids.shape[1]
        return self.embedding(ids) * self.scale + self.positions[:n]    # [B, n] -> [B, n, d]

In [7]:
torch.manual_seed(0)
src_embedding = TokenEmbedding(len(src_vocab), d_model, pad_id=PAD)
X = src_embedding(s)
print("s", list(s.shape), "-> X", list(X.shape))                   # [1, 5] -> [1, 5, 8]

E = src_embedding.embedding.weight                                 # [V_src, d] = [8, 8]
X_by_hand = math.sqrt(d_model) * E[s] + sinusoidal_positions(5, d_model)
assert torch.allclose(X, X_by_hand)                                # the module is exactly the formula
print("PAD row of E:", E[PAD].tolist())                            # zeros because of padding_idx
print("parameters:", [name for name, _ in src_embedding.named_parameters()])   # P is not a parameter
print("buffers:   ", [name for name, _ in src_embedding.named_buffers()])

s [1, 5] -> X [1, 5, 8]
PAD row of E: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
parameters: ['embedding.weight']
buffers:    ['positions']


### 思考题：绝对位置还是相对位置？

第 1 步把位置向量 $P_p$ **加**到 embedding 上。笔记本后面的 RoPE 一节会介绍另一种做法：不加 $P$，而是把 query 和 key 的特征两两配对，每一对看成平面上的一个点，位置 $p$ 的向量**旋转**一个角度 $p\,\omega_i$，$\omega_i$ 就是 1a 里那组频率。

注意力分数是第 2 步的内容，这里先把它看成 query 向量和 key 向量的点积 $q \cdot k$，其中 $q = x W_Q$，$k = x W_K$，$x$ 是那个位置的输入向量。

**问题：** 同样的两个词、同样相隔 3 个位置，把它们整体平移到句子的别处，例如从位置 $(2, 5)$ 挪到 $(10, 13)$，它们之间的注意力分数会不会变？加位置向量和旋转，两种做法分别是什么情况？

#### 答案

加位置向量的方式会变，所以它编码的是**绝对位置**；旋转的方式不变，所以它编码的是**相对位置**。

下面用同一对词向量、同一组 $W_Q$、$W_K$ 实际算一下。两个词的内容固定不变，只改变它们所在的位置。

In [ ]:
torch.manual_seed(0)
W_query, W_key = torch.randn(d_model, d_model), torch.randn(d_model, d_model)   # stand-ins for step 2's W_Q, W_K
e_q, e_k = torch.randn(d_model), torch.randn(d_model)                         # the content of the two words, fixed
P_long = sinusoidal_positions(200, d_model)


def turn(x, p):
    """Preview of RoPE: turn pair i of x by the angle p * omega_i. The RoPE section calls this rotate."""
    omega = 10000 ** (-torch.arange(0, d_model, 2, dtype=torch.float64) / d_model)   # the speeds of 1a
    cos, sin = (p * omega).cos().float(), (p * omega).sin().float()
    even, odd = x[0::2], x[1::2]
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1).flatten()


def score_added(p_q, p_k):                  # position added to the input, as in step 1
    return (((e_q + P_long[p_q]) @ W_query) @ ((e_k + P_long[p_k]) @ W_key)).item()


def score_rotated(p_q, p_k):                # position applied by turning q and k, as in RoPE
    return (turn(e_q @ W_query, p_q) @ turn(e_k @ W_key, p_k)).item()


print(f"{'positions':<12}{'distance':>9}{'added P':>11}{'rotated':>10}")
for p_q, p_k in [(2, 5), (10, 13), (100, 103), (2, 6)]:
    print(f"{str((p_q, p_k)):<12}{p_k - p_q:>9}{score_added(p_q, p_k):>11.3f}{score_rotated(p_q, p_k):>10.3f}")
assert math.isclose(score_rotated(2, 5), score_rotated(100, 103), abs_tol=1e-3)

positions    distance    added P   rotated
(2, 5)              3    -61.116   -12.073
(10, 13)            3    -29.432   -12.073
(100, 103)          3    -44.995   -12.073
(2, 6)              4    -53.143   -11.376


距离同样是 3，加法的分数随位置变来变去，旋转的分数完全相同。距离变成 4，旋转的分数才改变。

**加法为什么是绝对位置。** 记两个词的内容为 $e_q$ 和 $e_k$，位置为 $p$ 和 $p'$。位置加进输入之后，$q = (e_q + P_p) W_Q$，$k = (e_k + P_{p'}) W_K$，点积展开有四项：

$$
q \cdot k = \underbrace{e_q A\, e_k^{\top}}_{\text{content only}} + \underbrace{e_q A\, P_{p'}^{\top}}_{\text{only } p'} + \underbrace{P_p A\, e_k^{\top}}_{\text{only } p} + \underbrace{P_p A\, P_{p'}^{\top}}_{p \text{ and } p'}, \qquad A = W_Q W_K^{\top}
$$

- **中间两项各只含一个位置。** 比如 $p' = 5$ 和 $p' = 13$ 给出的值不同，所以平移会改变分数。模型看到的是"2 号位的词和 5 号位的词"，而不是"相距 3 的两个词"。
- **1a 验证过的性质在这里保不住。** $P_p \cdot P_{p'}$ 本身只取决于距离，但最后一项中间夹着 $A$，前面还多了两项交叉项。

**旋转为什么是相对位置。** 先看一对特征，也就是平面上的两个向量。它们的点积等于两者长度之积乘以夹角的余弦。位置 $p$ 的 query 转过 $p\,\omega$，位置 $p'$ 的 key 转过 $p'\omega$：

- **长度不变。** 旋转不改变向量的长短，内容信息原样保留。
- **夹角只多了 $(p' - p)\,\omega$。** 两个词同时往后挪 8 个位置，两个向量各自多转 $8\omega$，它们之间的夹角不变。

就像钟面上的两根指针：整个钟一起转，两根指针的夹角不变。写成公式，$R(\alpha)$ 表示转过角度 $\alpha$：

$$
\big(R(p\,\omega)\, q\big) \cdot \big(R(p'\omega)\, k\big) = q \cdot R\big((p' - p)\,\omega\big)\, k
$$

右边只剩 $p' - p$，单独的 $p$ 和 $p'$ 都消失了。$d$ 个特征有 $d/2$ 对，相当于 $d/2$ 根转速不同的指针：转得快的分辨近距离，转得慢的分辨远距离，但每一根都只看夹角。

**区别归结起来**

- **加法**：位置混进了向量的内容，和内容一起进入点积，绝对位置会残留在分数里。
- **旋转**：位置只体现为方向，点积只比较两个方向的夹角，绝对位置被抵消掉。

相对位置的好处是学到的规律可以在任何位置复用。比如"关注前一个词"这个规律，在第 3 位学会了，在第 300 位也一样成立。

## 2. 第 2 步：编码器多头自注意力

**输入** $X \in \mathbb{R}^{n \times d}$，**输出** $A \in \mathbb{R}^{n \times d}$。

每个头 $i = 1, \dots, h$ 把同一个 $X$ 投影三次：

$$
Q_i = X W_i^{Q}, \qquad K_i = X W_i^{K}, \qquad V_i = X W_i^{V} \qquad \in \mathbb{R}^{n \times d_k}, \qquad W_i^{Q}, W_i^{K}, W_i^{V} \in \mathbb{R}^{d \times d_k}
$$

每个 query 和每个 key 打分，按行（对 key）做 softmax，再用这些权重对 value 加权平均：

$$
\text{head}_i = \operatorname{softmax}\!\left(\frac{Q_i K_i^{\top}}{\sqrt{d_k}} + M\right) V_i \;\in \mathbb{R}^{n \times d_k}
$$

把各头沿特征维拼接起来，再乘一次 $W^{O}$ 把它们混合：

$$
A = \operatorname{MHA}(X, X, X;\, M) = \big[\, \text{head}_1 \,\Vert\, \cdots \,\Vert\, \text{head}_h \,\big]\, W^{O} \;\in \mathbb{R}^{n \times d}, \qquad W^{O} \in \mathbb{R}^{d \times d}
$$

$M$ 每行对应一个 query，每列对应一个 key：能读的位置是 $0$，不能读的是 $-\infty$。

记号 $\operatorname{MHA}(\cdot, \cdot, \cdot;\, M)$ 的三个参数依次是产生 $Q$、$K$、$V$ 的输入，分号后面是 mask；自注意力里三个都是 $X$。第 2、6、7 步用的是同一个 `MultiHeadAttention` 类，但各建一个实例，权重互不共享。三处的区别在于 $Q$、$K$、$V$ 从哪里来，以及用哪个 mask。

### 2a. 类定义

代码里没有为每个头单独建矩阵。一个 `Linear(d, d)` 把 $h$ 个 $W_i^{Q}$ 并排放在一起：输出的第 $i$ 段（$d_k$ 个特征）就是第 $i$ 个头。$K$、$V$ 也一样。所以"分头"只是一次 reshape。`attention_weights` 保存最近一次调用的 softmax 权重，方便查看。

In [8]:
class MultiHeadAttention(nn.Module):
    """Steps 2, 6 and 7.

    head_i = softmax(Q_i K_i^T / sqrt(d_k) + M) V_i
    MHA(query, key, value) = [head_1 || ... || head_h] W_O
    """

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must split evenly into heads"
        self.h, self.d_k = num_heads, d_model // num_heads
        # One Linear(d, d) holds the h per-head matrices W_i^Q side by side; same for K, V.
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.attention_weights = None  # [B, h, Lq, Lk] of the last call, kept for inspection

    def forward(self, query: Tensor, key: Tensor, value: Tensor, mask: Tensor = None) -> Tensor:
        # query [B, Lq, d]; key, value [B, Lk, d]; mask broadcasts to [B, h, Lq, Lk], True = may read.
        B, Lq, d = query.shape
        Lk = key.shape[1]
        q = self.w_q(query).view(B, Lq, self.h, self.d_k).transpose(1, 2)   # [B, h, Lq, d_k]
        k = self.w_k(key).view(B, Lk, self.h, self.d_k).transpose(1, 2)     # [B, h, Lk, d_k]
        v = self.w_v(value).view(B, Lk, self.h, self.d_k).transpose(1, 2)   # [B, h, Lk, d_k]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)              # [B, h, Lq, Lk]
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))               # blocked key -> weight 0
        weights = F.softmax(scores, dim=-1)                                 # each query row sums to 1
        self.attention_weights = weights
        heads = weights @ v                                                 # [B, h, Lq, d_k]
        concat = heads.transpose(1, 2).reshape(B, Lq, d)                    # [B, Lq, h * d_k] = [B, Lq, d]
        return self.w_o(concat)                                             # [B, Lq, d]

### 2b. 逐行拆开 `forward`

建一个模块，然后在 $X$ 上一行一行地执行 `forward` 里的代码，每行都看形状。编码器的 mask 只遮 PAD 的 key；这句话里没有 PAD，所以 mask 全是 `True`。

In [9]:
torch.manual_seed(1)
self_attention = MultiHeadAttention(d_model, num_heads)
src_mask = (s != PAD)[:, None, None, :]                            # [B, 1, 1, n]: True = this key may be read
print("src_mask", list(src_mask.shape), src_mask[0, 0, 0].tolist())

B, Lq, d = X.shape
Lk = X.shape[1]
h, d_k = self_attention.h, self_attention.d_k
print(f"B={B}  Lq={Lq}  Lk={Lk}  d={d}  h={h}  d_k={d_k}")

src_mask [1, 1, 1, 5] [True, True, True, True, True]
B=1  Lq=5  Lk=5  d=8  h=2  d_k=4


**投影并分头。** `w_q(X)` 的形状是 `[B, n, d]`。`view` 把最后一维 $d = 8$ 切成 $h \times d_k = 2 \times 4$。`transpose(1, 2)` 把"头"这一维挪到序列维前面，得到 `[B, h, n, d_k]`，之后的矩阵乘法会在 $(B, h)$ 上批量进行。

最后的循环验证：第 $i$ 个头的 $Q_i$ 正好等于 $X W_i^{Q}$。`w_q.weight` 的形状是 `[d, d]`，每 $d_k$ 行属于一个头：前 4 行是第 1 个头，后 4 行是第 2 个头。把某个头的这 $d_k$ 行（`[d_k, d]`）转置，就得到它的 $W_i^{Q} \in \mathbb{R}^{d \times d_k}$；要转置是因为 `nn.Linear` 计算的是 $X W^{\top}$。代码里循环变量 `i` 从 0 数，所以写成 `W_q[i * d_k:(i + 1) * d_k]`。

In [10]:
q_flat = self_attention.w_q(X)                                     # [B, n, d]
q_split = q_flat.view(B, Lq, h, d_k)                               # [B, n, h, d_k]
q = q_split.transpose(1, 2)                                        # [B, h, n, d_k]
k = self_attention.w_k(X).view(B, Lk, h, d_k).transpose(1, 2)      # [B, h, n, d_k]
v = self_attention.w_v(X).view(B, Lk, h, d_k).transpose(1, 2)      # [B, h, n, d_k]
print("w_q(X)", list(q_flat.shape), "-> view", list(q_split.shape), "-> transpose", list(q.shape))

W_q = self_attention.w_q.weight                                    # [d, d]; nn.Linear computes X @ W_q.T
for i in range(h):
    W_i = W_q[i * d_k:(i + 1) * d_k].T                             # W_i^Q: [d, d_k]
    assert torch.allclose(q[:, i], X @ W_i)
print("q[:, i] == X @ W_i^Q for every head: one Linear(d, d) holds the h matrices side by side")

w_q(X) [1, 5, 8] -> view [1, 5, 2, 4] -> transpose [1, 2, 5, 4]
q[:, i] == X @ W_i^Q for every head: one Linear(d, d) holds the h matrices side by side


**打分。** 每个头算 $Q_i K_i^{\top} / \sqrt{d_k}$，形状 `[n, d_k] × [d_k, n] = [n, n]`，第 $j$ 行是 query $j$ 对所有 key 的分数。缩放用的是 $\sqrt{d_k} = 2$，不是 $\sqrt{d}$。原因是点积由 $d_k$ 项相加，如果每项方差为 1，点积的方差就是 $d_k$；除以 $\sqrt{d_k}$ 把方差拉回 1，softmax 才不会一开始就饱和。

In [11]:
scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)                  # [B, h, n, n]
print("scores", list(scores.shape), "  head 0:")
print(scores[0, 0].detach())

scores [1, 2, 5, 5]   head 0:
tensor([[  1.30,   0.25,   0.22,   0.56,   3.15],
        [  0.59,  -0.49,   1.25,   0.49,   0.51],
        [ -0.03,   0.13,  -1.72,   0.80,   4.19],
        [ -0.81,  -0.75,   1.68,  -0.57,  -5.03],
        [ -5.39,  -2.63,   0.86,  -0.58, -10.14]])


**mask。** `masked_fill(~mask, -inf)` 把不能读的位置设成 $-\infty$。softmax 之后 $e^{-\infty} = 0$，这些 key 的权重正好是 0，剩下的 key 在彼此之间重新归一化。mask 的形状 `[B, 1, 1, n]` 会广播到 `[B, h, n, n]`：所有头、所有 query 行共用同一个 mask，它只按列（key）遮。

这句话的 mask 全是 `True`，所以下面那行 `masked_fill` 在这里什么都不改变。下面用书里的数字单独演示一次：分数 $(2.0, 1.0, 0.5, 1.7, 3.0)$，最后两个 key 被遮住。用 $0$ 代替 $-\infty$ 是错的，因为 $e^{0} = 1$ 仍然会分到权重。

In [12]:
scores = scores.masked_fill(~src_mask, float("-inf"))              # no-op here: the sentence has no PAD

demo_scores = torch.tensor([2.0, 1.0, 0.5, 1.7, 3.0])
demo_mask = torch.tensor([True, True, True, False, False])         # the last two keys are blocked
print("with -inf:", F.softmax(demo_scores.masked_fill(~demo_mask, float("-inf")), dim=-1))
print("with 0:   ", F.softmax(demo_scores.masked_fill(~demo_mask, 0.0), dim=-1))   # wrong: blocked keys still get weight

with -inf: tensor([0.63, 0.23, 0.14, 0.00, 0.00])
with 0:    tensor([0.54, 0.20, 0.12, 0.07, 0.07])


**softmax。** 对最后一维（key）做 softmax，每个 query 行的权重加起来等于 1。

In [13]:
weights = F.softmax(scores, dim=-1)                                # [B, h, n, n]
print("weights, head 0:")
print(weights[0, 0].detach())
print("row sums:", weights.sum(dim=-1)[0, 0].detach())

weights, head 0:
tensor([[    0.12,     0.04,     0.04,     0.06,     0.75],
        [    0.20,     0.07,     0.38,     0.18,     0.18],
        [    0.01,     0.02,     0.00,     0.03,     0.94],
        [    0.06,     0.07,     0.78,     0.08,     0.00],
        [    0.00,     0.02,     0.79,     0.19,     0.00]])
row sums: tensor([1.00, 1.00, 1.00, 1.00, 1.00])


**加权平均 value，拼接各头，乘 $W^{O}$。** 每个头的 `weights @ v` 是 `[n, n] × [n, d_k] = [n, d_k]`。拼接 $[\text{head}_1 \Vert \text{head}_2]$ 的做法是：先 `transpose(1, 2)` 把头挪回序列维后面，再 `reshape` 把 $(h, d_k)$ 合并成 $d$。这里必须用 `reshape` 而不是 `view`，因为 transpose 之后内存不再连续。

In [14]:
heads = weights @ v                                                # [B, h, n, d_k]
concat = heads.transpose(1, 2).reshape(B, Lq, d)                   # [B, n, h * d_k] = [B, n, d]
A_by_hand = self_attention.w_o(concat)                             # [B, n, d]
print("heads", list(heads.shape), "-> concat", list(concat.shape), "-> A", list(A_by_hand.shape))
assert torch.allclose(concat, torch.cat([heads[:, i] for i in range(h)], dim=-1))   # [head_1 || head_2]

heads [1, 2, 5, 4] -> concat [1, 5, 8] -> A [1, 5, 8]


### 2c. 核对：逐行算的结果等于模块的输出

调用一次模块，结果应该和上面逐行算的一样。再和 PyTorch 自带的 `F.scaled_dot_product_attention` 对比各头的输出，它的布尔 mask 也是 `True` = 可以读。

In [15]:
A = self_attention(X, X, X, src_mask)
assert torch.allclose(A, A_by_hand)
assert torch.allclose(self_attention.attention_weights, weights)
assert torch.allclose(heads, F.scaled_dot_product_attention(q, k, v, attn_mask=src_mask), atol=1e-6)
print("X", list(X.shape), "-> A", list(A.shape), "   weights per head", list(self_attention.attention_weights.shape))

X [1, 5, 8] -> A [1, 5, 8]    weights per head [1, 2, 5, 5]


**为什么第 1 步要加位置。** 注意力本身不看顺序。下面把词序打乱成 `video a we watch short`：

- **不加 $P$**：输出只是跟着换了顺序，每个词得到的结果完全不变。
- **加上 $P$**：每个词拿到它新位置的那一行位置向量，每个词得到的结果本身也变了，不再只是换了顺序。

所以顺序信息只能来自 $P$（或者后面的 RoPE）。

In [16]:
perm = torch.tensor([4, 2, 0, 1, 3])                               # shuffle the 5 words
s_perm = s[:, perm]
print("shuffled sentence:", words(s_perm[0], src_vocab))

X_noP = math.sqrt(d_model) * src_embedding.embedding(s)            # step 1 without P
X_noP_perm = math.sqrt(d_model) * src_embedding.embedding(s_perm)
A_noP = self_attention(X_noP, X_noP, X_noP, src_mask)
A_noP_perm = self_attention(X_noP_perm, X_noP_perm, X_noP_perm, src_mask)
assert torch.allclose(A_noP_perm, A_noP[:, perm], atol=1e-6)       # the same outputs, only reordered
print("without P: outputs only reordered?", torch.allclose(A_noP_perm, A_noP[:, perm], atol=1e-6))

X_perm = src_embedding(s_perm)                                     # with P: each word gets the P row of its new position
A_perm = self_attention(X_perm, X_perm, X_perm, src_mask)
print("with P:    outputs only reordered?", torch.allclose(A_perm, A[:, perm], atol=1e-6))

shuffled sentence: video a we watch short
without P: outputs only reordered? True
with P:    outputs only reordered? False


## 3. 第 3 步：Add & Norm

**输入** $X, A \in \mathbb{R}^{n \times d}$，**输出** $X_1 \in \mathbb{R}^{n \times d}$。

$$
X_1 = \operatorname{LN}(X + A), \qquad \operatorname{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta, \qquad \mu = \frac{1}{d}\sum_{k=0}^{d-1} x_k, \quad \sigma^2 = \frac{1}{d}\sum_{k=0}^{d-1} (x_k - \mu)^2
$$

- **Add（残差）。** 子层只需要学习对 $X$ 的修正量，梯度也能沿着这条加法直接流回去。
- **Norm。** LayerNorm 对每个 token 的 $d$ 个特征单独求均值和方差，5 个 token 就有 5 组统计量。$\gamma, \beta \in \mathbb{R}^{d}$ 是可学习的，所有 token 共用。它不跨 token、也不跨 batch 求平均，所以变长序列和 batch size 1 都没问题，这一点和 BatchNorm 不同。现在的大模型多用 RMSNorm：在同一个维度上归一化，但不减均值，直接除以均方根 $\sqrt{\tfrac{1}{d}\sum_k x_k^2 + \epsilon}$，也没有 $\beta$。

这一步没有新类，直接用 `nn.LayerNorm`。手算一遍核对：

In [2]:
norm1 = nn.LayerNorm(d_model)                                      # gamma = 1, beta = 0 at initialization
X1 = norm1(X + A)

r = X + A
mu = r.mean(dim=-1, keepdim=True)                                  # [B, n, 1]: one mean per token
var = r.var(dim=-1, unbiased=False, keepdim=True)                  # [B, n, 1]: divide by d, not d - 1
X1_by_hand = norm1.weight * (r - mu) / torch.sqrt(var + norm1.eps) + norm1.bias
assert torch.allclose(X1, X1_by_hand, atol=1e-6)
print("X1", list(X1.shape))
print("per-token mean:", X1.mean(dim=-1)[0].detach())
print("per-token std: ", X1.std(dim=-1, unbiased=False)[0].detach())

NameError: name 'nn' is not defined

## 4. 第 4 步：前馈网络 FFN + Add & Norm

**输入** $X_1 \in \mathbb{R}^{n \times d}$，**输出** $H \in \mathbb{R}^{n \times d}$。

$$
\operatorname{FFN}(x) = \operatorname{ReLU}(x W_1 + b_1)\, W_2 + b_2, \qquad W_1 \in \mathbb{R}^{d \times d_{ff}}, \quad W_2 \in \mathbb{R}^{d_{ff} \times d}
$$

$$
H = \operatorname{LN}\big(X_1 + \operatorname{FFN}(X_1)\big) \;\in \mathbb{R}^{n \times d}
$$

- **逐位置。** 小写 $x$ 是一个 token 的向量，即 $X_1$ 的一行。同一组 $W_1, b_1, W_2, b_2$ 作用在每一行上，位置之间没有任何信息交换；只有注意力在 token 之间搬运信息。
- **先变宽再变回来。** 形状 $[5, 8] \to [5, 32] \to [5, 8]$：变宽、过非线性、回到 $d$，残差加法才能对上。
- **分工。** 注意力决定读哪些 token，FFN 加工读到的内容。它有 $2\,d\,d_{ff}$ 个权重，大约占一层参数的三分之二。

In [18]:
class FeedForward(nn.Module):
    """FFN(x) = ReLU(x W_1 + b_1) W_2 + b_2, applied to every position independently."""

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)   # W_1: d -> d_ff
        self.linear2 = nn.Linear(d_ff, d_model)   # W_2: d_ff -> d

    def forward(self, x: Tensor) -> Tensor:
        return self.linear2(F.relu(self.linear1(x)))   # [B, L, d] -> [B, L, d_ff] -> [B, L, d]

In [19]:
torch.manual_seed(2)
ffn = FeedForward(d_model, d_ff)
hidden = F.relu(ffn.linear1(X1))                                   # [B, n, d_ff]
print("X1", list(X1.shape), "-> hidden", list(hidden.shape), "-> FFN", list(ffn(X1).shape))
print(f"fraction of hidden units set to 0 by ReLU: {(hidden == 0).float().mean().item():.2f}")

# Position-wise: token 2 on its own gives the same output as token 2 inside the sentence.
assert torch.allclose(ffn(X1)[:, 2], ffn(X1[:, 2:3])[:, 0])

norm2 = nn.LayerNorm(d_model)
H = norm2(X1 + ffn(X1))                                            # the encoder output, called memory
print("H", list(H.shape))

n_attention = sum(p.numel() for p in self_attention.parameters())
n_ffn = sum(p.numel() for p in ffn.parameters())
print(f"weights: attention {n_attention} = 4 d^2,  FFN {n_ffn} = 2 d d_ff + d_ff + d")

X1 [1, 5, 8] -> hidden [1, 5, 32] -> FFN [1, 5, 8]
fraction of hidden units set to 0 by ReLU: 0.54
H [1, 5, 8]
weights: attention 256 = 4 d^2,  FFN 552 = 2 d d_ff + d_ff + d


### 组装：`EncoderLayer` = 第 2–4 步

`forward` 的两行就是两个 Add & Norm：

$$
X_1 = \operatorname{LN}\big(X + \operatorname{MHA}(X, X, X)\big), \qquad H = \operatorname{LN}\big(X_1 + \operatorname{FFN}(X_1)\big)
$$

先加残差再做 LN，这就是 post-norm。现在的大模型多用 pre-norm，$x + \operatorname{Sublayer}(\operatorname{LN}(x))$，深层网络更容易训练。叠 $N$ 层时，第 2–4 步用各自的权重重复 $N$ 次，$H$ 是最后一层的输出。

In [20]:
class EncoderLayer(nn.Module):
    """Steps 2-4:  X1 = LN(X + MHA(X, X, X));  H = LN(X1 + FFN(X1))."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)   # normalizes the d features of each token separately
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: Tensor, src_mask: Tensor) -> Tensor:
        x = self.norm1(x + self.self_attention(x, x, x, src_mask))
        return self.norm2(x + self.ffn(x))

把上面手动建好的四个模块装进一个 `EncoderLayer`。如果类和公式一致，它的输出应该正好等于我们一步步算出的 $H$。

In [21]:
encoder_layer = EncoderLayer(d_model, num_heads, d_ff)
encoder_layer.self_attention, encoder_layer.ffn = self_attention, ffn   # reuse our modules
encoder_layer.norm1, encoder_layer.norm2 = norm1, norm2
assert torch.allclose(encoder_layer(X, src_mask), H)
print("EncoderLayer(X) == H, shape", list(H.shape))

EncoderLayer(X) == H, shape [1, 5, 8]


## 5. 第 5 步：解码器输入

**输入** $\mathbf{t} \in \{0,\dots,V_{\text{tgt}}-1\}^{m}$，**输出** $Y \in \mathbb{R}^{m \times d}$。

$$
Y = \sqrt{d}\; E_{\text{tgt}}[\mathbf{t}] + P_{0:m-1}, \qquad E_{\text{tgt}} \in \mathbb{R}^{V_{\text{tgt}} \times d}
$$

就是第 1 步再来一次：目标语言用自己的表，位置编码 $P$ 相同。

In [22]:
torch.manual_seed(3)
tgt_embedding = TokenEmbedding(len(tgt_vocab), d_model, pad_id=PAD)
Y = tgt_embedding(t)
print("t", list(t.shape), "-> Y", list(Y.shape))                   # [1, 4] -> [1, 4, 8]

t [1, 4] -> Y [1, 4, 8]


### 两种 mask

解码器要用到两种 mask。它们都是布尔张量（`True` = 可以读），形状都能广播到 `[B, h, L_q, L_k]`：

- **源 mask $M^{\text{pad}}$**：`(src_ids != PAD)[:, None, None, :]`，形状 `[B, 1, 1, S]`，$S = n$ 是源句长度。它只遮 PAD 的 key（列），所有 query 行都一样。第 2 步的编码器自注意力和第 7 步的交叉注意力用它。
- **目标 mask**：下三角矩阵 `tril` 保证 query $j$ 只读 key $k \le j$，再和"key 不是 PAD"取与。形状 `[B, 1, T, T]`，$T = m$ 是目标句长度。第 6 步用它。

$$
M^{\text{causal}}_{jk} = \begin{cases} 0 & k \le j \\ -\infty & k > j \end{cases}
$$

在 `Transformer` 类里，它们是 `source_mask` 和 `target_mask` 两个方法。这里先用同样的表达式直接算：

In [23]:
T = t.shape[1]
causal = torch.ones(T, T, dtype=torch.bool).tril()                 # key k <= query j
tgt_mask = causal[None, None] & (t != PAD)[:, None, None, :]       # [B, 1, T, T]
print("tgt_mask", list(tgt_mask.shape), " rows = queries, columns = keys, 1 = may read")
print(tgt_mask[0, 0].int())

tgt_mask [1, 1, 4, 4]  rows = queries, columns = keys, 1 = may read
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)


例句里没有 PAD，看不出 PAD 那一半的效果。换一个补齐过的 batch：第二句比第一句短，末尾补了 PAD。

In [24]:
src_batch = torch.tensor([[3, 4, 5, 6, 7],
                          [3, 4, 7, 0, 0]])                        # "we watch video" + 2 PAD
tgt_batch = torch.tensor([[1, 3, 4, 5],
                          [1, 3, 4, 0]])                           # "BOS nous regardons" + 1 PAD

src_mask_batch = (src_batch != PAD)[:, None, None, :]              # [2, 1, 1, 5]
print("source mask", list(src_mask_batch.shape), " sentence 2:", src_mask_batch[1, 0, 0].int().tolist())

causal_batch = torch.ones(4, 4, dtype=torch.bool).tril()
tgt_mask_batch = causal_batch[None, None] & (tgt_batch != PAD)[:, None, None, :]   # [2, 1, 4, 4]
print("target mask", list(tgt_mask_batch.shape), " sentence 2 (causal AND key is not PAD):")
print(tgt_mask_batch[1, 0].int())

source mask [2, 1, 1, 5]  sentence 2: [1, 1, 1, 0, 0]
target mask [2, 1, 4, 4]  sentence 2 (causal AND key is not PAD):
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 0]], dtype=torch.int32)


- **PAD 那一列整列为 0。** 没有任何 query 能读 PAD key。
- **PAD 那一行仍然能读前面的 key。** 第 0 列是 BOS，永远不是 PAD，而因果 mask 让每一行 $j$ 都能读 $k = 0 \le j$ 这一列。所以不会出现整行都是 $-\infty$ 的情况，那样 softmax 会得到 NaN。PAD 那一行的输出没人用：它的标签是 PAD，第 10 步的 loss 会忽略它。
- **这依赖于 PAD 补在右边。** mask 本身并不能防止整行被遮。如果把 PAD 补在左边，例如 `[0, 1, 3, 4]`，因果 mask 只允许第 0 行读第 0 列，而这一列是 PAD，又被遮掉，于是第 0 行什么都读不到，整行都是 $-\infty$，softmax 输出 NaN（见下面的输出）。源端没有这个问题：源 mask 不带因果部分，每一行都能读所有非 PAD 的列，只要句子里至少有一个真实的词，PAD 补在哪一边都不会出现整行被遮。

In [25]:
tgt_left = torch.tensor([[0, 1, 3, 4]])                            # PAD on the LEFT: a layout this model does not support
mask_left = causal_batch[None, None] & (tgt_left != PAD)[:, None, None, :]
print("target mask with left padding:")
print(mask_left[0, 0].int())                                       # row 0 can read nothing
row0 = torch.zeros(4).masked_fill(~mask_left[0, 0, 0], float("-inf"))
print("softmax of row 0:", F.softmax(row0, dim=-1))                # all -inf -> NaN

target mask with left padding:
tensor([[0, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 1, 1, 0],
        [0, 1, 1, 1]], dtype=torch.int32)
softmax of row 0: tensor([nan, nan, nan, nan])


## 6. 第 6 步：掩码自注意力 + Add & Norm

**输入** $Y \in \mathbb{R}^{m \times d}$，**输出** $Y_1 \in \mathbb{R}^{m \times d}$。

$$
Y_1 = \operatorname{LN}\big(Y + \operatorname{MHA}(Y, Y, Y;\, M^{\text{causal}})\big)
$$

和第 2 步是同一个 `MultiHeadAttention` 类，有自己的一套权重，每个头的分数是 `[4, 4]`。第 $j$ 行只能读 $k \le j$ 的 key，所以位置 $j$ 的预测只依赖 $t_0, \dots, t_j$（$t_0$ 是 BOS），看不到它的标签 $y_j = t_{j+1}$。

打印出的权重里，对角线以上一定是 0，这是 mask 的作用。对角线及以下的数值来自随机初始化，某一行把几乎全部权重放在一个 key 上也很正常。

In [26]:
torch.manual_seed(4)
decoder_self_attention = MultiHeadAttention(d_model, num_heads)
decoder_norm1 = nn.LayerNorm(d_model)
Y1 = decoder_norm1(Y + decoder_self_attention(Y, Y, Y, tgt_mask))
print("Y", list(Y.shape), "-> Y1", list(Y1.shape))
print("causal weights, head 0 (rows sum to 1, zeros above the diagonal):")
print(decoder_self_attention.attention_weights[0, 0].detach())

Y [1, 4, 8] -> Y1 [1, 4, 8]
causal weights, head 0 (rows sum to 1, zeros above the diagonal):
tensor([[    1.00,     0.00,     0.00,     0.00],
        [    1.00,     0.00,     0.00,     0.00],
        [    0.43,     0.30,     0.27,     0.00],
        [    0.15,     0.53,     0.22,     0.10]])


**验证因果性。** 把最后一个输入 token 从 `video` 换成 `une`。前 3 个位置看不到它，输出完全不变；只有最后一个位置变了。

In [27]:
t_changed = t.clone()
t_changed[0, 3] = tgt_vocab.index("une")
Y_changed = tgt_embedding(t_changed)
Y1_changed = decoder_norm1(Y_changed + decoder_self_attention(Y_changed, Y_changed, Y_changed, tgt_mask))
print("rows 0-2 unchanged:", torch.allclose(Y1[:, :3], Y1_changed[:, :3]))
print("row 3 changed:     ", not torch.allclose(Y1[:, 3], Y1_changed[:, 3]))

rows 0-2 unchanged: True
row 3 changed:      True


## 7. 第 7 步：交叉注意力 + Add & Norm

**输入** $Y_1 \in \mathbb{R}^{m \times d}$ 和编码器输出 $H \in \mathbb{R}^{n \times d}$，**输出** $Y_2 \in \mathbb{R}^{m \times d}$。

$$
Y_2 = \operatorname{LN}\big(Y_1 + \operatorname{MHA}(Y_1, H, H;\, M^{\text{pad}})\big), \qquad Q_i = Y_1 W_i^{Q}, \quad K_i = H W_i^{K}, \quad V_i = H W_i^{V}
$$

- **query 来自目标端，key 和 value 来自源端。** 这是解码器唯一读取源句的地方。
- **形状。** 每个头的分数是 `[4, d_k] × [d_k, 5] = [4, 5]`，行是目标位置，列是源位置。输出长度跟着 query 走，所以仍是 `[4, 8]`。
- **只用源 PAD mask，不用因果 mask。** 整个源句都是已知的，没有"未来"需要遮。

In [28]:
torch.manual_seed(5)
cross_attention = MultiHeadAttention(d_model, num_heads)
decoder_norm2 = nn.LayerNorm(d_model)
Y2 = decoder_norm2(Y1 + cross_attention(Y1, H, H, src_mask))       # Q from Y1; K, V from H
print("Y1", list(Y1.shape), "+ H", list(H.shape), "-> Y2", list(Y2.shape))
print("cross weights, head 0 (rows = target tokens, columns = source tokens):")
print(cross_attention.attention_weights[0, 0].detach())

Y1 [1, 4, 8] + H [1, 5, 8] -> Y2 [1, 4, 8]
cross weights, head 0 (rows = target tokens, columns = source tokens):
tensor([[0.19, 0.19, 0.15, 0.25, 0.22],
        [0.22, 0.17, 0.28, 0.17, 0.16],
        [0.22, 0.21, 0.21, 0.17, 0.18],
        [0.18, 0.17, 0.26, 0.20, 0.19]])


## 8. 第 8 步：FFN + Add & Norm，组装 `DecoderLayer`

**输入** $Y_2 \in \mathbb{R}^{m \times d}$，**输出** $Y_3 \in \mathbb{R}^{m \times d}$。

$$
Y_3 = \operatorname{LN}\big(Y_2 + \operatorname{FFN}(Y_2)\big)
$$

和第 4 步形式相同，权重独立：$[4, 8] \to [4, 32] \to [4, 8]$。

In [29]:
torch.manual_seed(6)
decoder_ffn = FeedForward(d_model, d_ff)
decoder_norm3 = nn.LayerNorm(d_model)
Y3 = decoder_norm3(Y2 + decoder_ffn(Y2))
print("Y2", list(Y2.shape), "-> Y3", list(Y3.shape))

Y2 [1, 4, 8] -> Y3 [1, 4, 8]


### 组装：`DecoderLayer` = 第 6–8 步

`forward` 的三行对应三个 Add & Norm：掩码自注意力、交叉注意力、FFN。叠 $N$ 层时，每一层的交叉注意力读的都是同一个 $H$。

In [30]:
class DecoderLayer(nn.Module):
    """Steps 6-8:  Y1 = LN(Y + MHA(Y, Y, Y; causal));  Y2 = LN(Y1 + MHA(Y1, H, H; pad));
    Y3 = LN(Y2 + FFN(Y2))."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, y: Tensor, memory: Tensor, tgt_mask: Tensor, src_mask: Tensor) -> Tensor:
        y = self.norm1(y + self.self_attention(y, y, y, tgt_mask))
        y = self.norm2(y + self.cross_attention(y, memory, memory, src_mask))  # Q from y; K, V from H
        return self.norm3(y + self.ffn(y))

In [31]:
decoder_layer = DecoderLayer(d_model, num_heads, d_ff)
decoder_layer.self_attention, decoder_layer.cross_attention = decoder_self_attention, cross_attention
decoder_layer.ffn = decoder_ffn
decoder_layer.norm1, decoder_layer.norm2, decoder_layer.norm3 = decoder_norm1, decoder_norm2, decoder_norm3
assert torch.allclose(decoder_layer(Y, H, tgt_mask, src_mask), Y3)
print("DecoderLayer(Y, H) == Y3, shape", list(Y3.shape))

DecoderLayer(Y, H) == Y3, shape [1, 4, 8]


## 9. 第 9 步：logits，组装整个 `Transformer`

**输入** $Y_3 \in \mathbb{R}^{m \times d}$，**输出** $Z \in \mathbb{R}^{m \times V_{\text{tgt}}}$。

$$
Z = Y_3 W^{\text{vocab}} + b^{\text{vocab}}, \qquad W^{\text{vocab}} \in \mathbb{R}^{d \times V_{\text{tgt}}}, \qquad p\big(t_{j+1} = v \,\big|\, t_{0:j}, \mathbf{s}\big) = \operatorname{softmax}(Z_j)_v
$$

第 $j$ 行给词表里的每个词打分，表示它作为前缀 $t_{0:j}$ 之后下一个词的可能性。最后一维现在索引的是词，不再是特征；真实词表下它会是 `[4, 32000]` 这样的形状。

模型还没训练，下面打印的"最可能的下一个词"只是随机权重给出的结果，出现 `<pad>` 也不奇怪。第 10 步的 loss 衡量的正是它离正确答案 `nous regardons video <eos>` 有多远。

In [32]:
torch.manual_seed(7)
vocab_projection = nn.Linear(d_model, len(tgt_vocab))              # W_vocab: d -> V_tgt
Z = vocab_projection(Y3)
print("Y3", list(Y3.shape), "-> Z", list(Z.shape))
print("row sums of softmax(Z):", F.softmax(Z, dim=-1).sum(dim=-1)[0].detach())
for j in range(Z.shape[1]):
    print(f"position {j} ({tgt_vocab[t[0, j]]:>9}): most likely next word = {tgt_vocab[Z[0, j].argmax()]}")

Y3 [1, 4, 8] -> Z [1, 4, 10]
row sums of softmax(Z): tensor([1.00, 1.00, 1.00, 1.00])
position 0 (    <bos>): most likely next word = tu
position 1 (     nous): most likely next word = tu
position 2 (regardons): most likely next word = regardes
position 3 (    video): most likely next word = <pad>


### 组装：`Transformer`

- **`__init__`**：两个 embedding，`num_layers` 个编码器层和解码器层（`nn.ModuleList`），以及输出投影。
- **`source_mask` / `target_mask`**：第 5 步那两种 mask。
- **`encode`**：第 1–4 步，返回 memory $H$。
- **`decode`**：第 5–9 步，返回 logits $Z$。
- **`forward`**：先 `encode` 再 `decode`。

In [33]:
class Transformer(nn.Module):
    """Encoder(s) -> memory H;  Decoder(t, H) -> logits Z.  Steps 1-9 end to end."""

    def __init__(self, src_vocab: int, tgt_vocab: int, d_model: int = 8, num_heads: int = 2,
                 d_ff: int = 32, num_layers: int = 1, pad_id: int = 0, max_len: int = 512):
        super().__init__()
        self.pad_id = pad_id
        self.src_embedding = TokenEmbedding(src_vocab, d_model, pad_id, max_len)
        self.tgt_embedding = TokenEmbedding(tgt_vocab, d_model, pad_id, max_len)
        self.encoder = nn.ModuleList(EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers))
        self.decoder = nn.ModuleList(DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers))
        self.vocab_projection = nn.Linear(d_model, tgt_vocab)   # W_vocab: d -> V_tgt

    def source_mask(self, src_ids: Tensor) -> Tensor:
        return (src_ids != self.pad_id)[:, None, None, :]        # [B, 1, 1, S]: PAD keys are hidden

    def target_mask(self, tgt_ids: Tensor) -> Tensor:
        T = tgt_ids.shape[1]
        causal = torch.ones(T, T, dtype=torch.bool, device=tgt_ids.device).tril()  # key k <= query j
        return causal[None, None] & (tgt_ids != self.pad_id)[:, None, None, :]     # [B, 1, T, T]

    def encode(self, src_ids: Tensor) -> Tensor:
        src_mask = self.source_mask(src_ids)
        x = self.src_embedding(src_ids)                          # step 1:    [B, S] -> [B, S, d]
        for layer in self.encoder:
            x = layer(x, src_mask)                               # steps 2-4: [B, S, d]
        return x                                                 # memory H:  [B, S, d]

    def decode(self, tgt_ids: Tensor, memory: Tensor, src_mask: Tensor) -> Tensor:
        tgt_mask = self.target_mask(tgt_ids)
        y = self.tgt_embedding(tgt_ids)                          # step 5:    [B, T] -> [B, T, d]
        for layer in self.decoder:
            y = layer(y, memory, tgt_mask, src_mask)             # steps 6-8: [B, T, d]
        return self.vocab_projection(y)                          # step 9:    logits Z [B, T, V_tgt]

    def forward(self, src_ids: Tensor, tgt_ids: Tensor) -> Tensor:
        return self.decode(tgt_ids, self.encode(src_ids), self.source_mask(src_ids))

同样把手动建好的模块装进去。如果类和公式一致，`model.encode(s)` 应该等于 $H$，`model(s, t)` 应该正好等于我们一步步算出的 $Z$。

In [34]:
model = Transformer(len(src_vocab), len(tgt_vocab), d_model=d_model, num_heads=num_heads, d_ff=d_ff)
model.src_embedding, model.tgt_embedding = src_embedding, tgt_embedding
model.encoder[0], model.decoder[0] = encoder_layer, decoder_layer
model.vocab_projection = vocab_projection

assert torch.equal(model.source_mask(s), src_mask) and torch.equal(model.target_mask(t), tgt_mask)
assert torch.allclose(model.encode(s), H)
assert torch.allclose(model(s, t), Z)
print("model.encode(s) == H and model(s, t) == Z, shape", list(model(s, t).shape))
print("parameters:", sum(p.numel() for p in model.parameters()))

model.encode(s) == H and model(s, t) == Z, shape [1, 4, 10]
parameters: 2186


## 10. 第 10 步：loss 与一次参数更新

**输入** $Z \in \mathbb{R}^{m \times V_{\text{tgt}}}$ 和标签 $\mathbf{y}$，**输出** 标量 $\mathcal{L}$。

$$
\mathcal{L} = -\frac{1}{m} \sum_{j=0}^{m-1} \log \operatorname{softmax}(Z_j)_{y_j} = -\frac{1}{m} \sum_{j=0}^{m-1} \Big( Z_{j,\, y_j} - \log \sum_{v=0}^{V_{\text{tgt}}-1} e^{Z_{j,\, v}} \Big)
$$

- **每个位置一个"下一个词"的交叉熵。** 标签是 PAD 的位置不计入平均，这就是 `ignore_index`。
- **`F.cross_entropy` 接收原始的 $Z$。** 它自己做 log-softmax，所以不要先做 softmax。它把 `dim=1`（从 0 数，也就是第二个维度）当作类别维：输入 `[N, V]` 配标签 `[N]`。$Z$ 的词表维在最后，直接传 `[B, m, V]` 的话，$m$ 会被当成类别数：这里会报错 `Expected target size [1, 10], got [1, 4]`，碰巧 $m = V$ 时不报错，但算出来的是错的。所以先把 batch 和位置两维展平成 `[B·m, V]`，标签展平成 `[B·m]`。
- **初始 loss 和 $\log 10 = 2.30$ 在同一量级。** 随机初始化的模型对 10 个词差不多一视同仁；logits 不完全相等，所以实际值会略高一些。

In [35]:
loss = F.cross_entropy(Z.reshape(-1, len(tgt_vocab)), y.reshape(-1), ignore_index=PAD)
log_probs = F.log_softmax(Z, dim=-1)                               # [B, m, V]
loss_by_hand = -log_probs[0, torch.arange(4), y[0]].mean()         # log p(y_j), picked from each row
assert torch.allclose(loss, loss_by_hand)
print(f"loss = {loss.item():.4f}   (log 10 = {math.log(10):.4f})")

loss = 2.5487   (log 10 = 2.3026)


**反向传播与一步更新。** `loss.backward()` 算出每个参数的 $\partial \mathcal{L} / \partial \theta$，编码器的参数也包括在内：梯度经过交叉注意力的 $K$、$V$ 流回 $H$，再流回第 1–4 步。`optimizer.step()` 用这些梯度把所有参数更新一次。源 embedding 表里只有句子里出现过的行（id 3–7）拿到了梯度。

**注意：** `model` 里的模块就是前面各节建的那些对象，`optimizer.step()` 会原地修改它们的权重。运行这一格之后，如果单独重跑第 2–9 节里的核对格，会因为 $A$、$H$、$Z$ 还是更新前的值而 assert 失败。要重新核对，请重启 kernel 后从头运行全部单元。

In [36]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad()
loss = F.cross_entropy(model(s, t).reshape(-1, len(tgt_vocab)), y.reshape(-1), ignore_index=PAD)   # fresh graph, so this cell can run again
loss.backward()                                                    # dL/dtheta for every parameter, encoder included
grad_rows = model.src_embedding.embedding.weight.grad.abs().sum(dim=1)
print("source embedding rows with a gradient:", grad_rows.nonzero().flatten().tolist())
print("encoder W_Q has a gradient:", model.encoder[0].self_attention.w_q.weight.grad.abs().sum().item() > 0)
optimizer.step()                                                   # one update; a real model needs many thousands
loss_after = F.cross_entropy(model(s, t).reshape(-1, len(tgt_vocab)), y.reshape(-1), ignore_index=PAD)
print(f"loss before {loss.item():.4f} -> after one step {loss_after.item():.4f}")

source embedding rows with a gradient: [3, 4, 5, 6, 7]
encoder W_Q has a gradient: True
loss before 2.5487 -> after one step 2.5091


## 11. 第 11 步：贪心生成

**输入** 源句 $\mathbf{s}$，**输出** 生成的序列 $\hat{\mathbf{t}}$。编码器只跑一次，然后从 BOS 开始，每次取分数最高的词接到末尾，遇到 EOS 或者达到长度上限就停：

$$
H = \operatorname{Encoder}(\mathbf{s}), \qquad \hat t_0 = \text{BOS}, \qquad \hat t_{j+1} = \arg\max_{v}\ Z_j\big(\hat t_{0:j},\, H\big)_v
$$

- **每一步都在当前前缀上重跑第 5–9 步**，只用最后一行 $Z_j$。前缀从 `[1]` 长到 `[2]`、`[3]`……，$H$ 一直是 `[5, 8]`。
- **`finished`** 记录 batch 里哪些句子已经生成过 EOS，之后它们只补 PAD；全部结束就提前退出。
- **代价。** 前缀 $\hat t_{0:j}$ 有 $j+1$ 个位置，每步都把它们全部重算：投影和 FFN 是 $O(j)$，自注意力的分数是 $(j+1) \times (j+1)$，即 $O(j^2)$。书中 Inference 一章的 KV cache 把过去位置的 $K_i, V_i$ 存起来，每步只算新位置这一行：投影和 FFN 降到 $O(1)$，注意力仍要读全部 $j+1$ 个 key，是 $O(j)$。
- **采样。** 把 argmax 换成按概率抽样（temperature、top-k、top-p）就是采样解码。
- **`@torch.no_grad()` 和 `model.eval()`。** 生成时不需要梯度，`no_grad` 不建计算图，省内存和时间。`eval()` 把 dropout、BatchNorm 等切到推理模式；这个模型两者都没有，所以 `eval()` 在这里不改变结果，只是推理前的标准写法。

In [37]:
@torch.no_grad()
def greedy_decode(model: Transformer, src_ids: Tensor, bos_id: int = 1, eos_id: int = 2,
                  max_new_tokens: int = 8) -> Tensor:
    """Step 11: encode once, then append argmax_v Z_j until every row has produced EOS."""
    memory, src_mask = model.encode(src_ids), model.source_mask(src_ids)
    ids = torch.full((src_ids.shape[0], 1), bos_id, dtype=torch.long, device=src_ids.device)
    finished = torch.zeros(src_ids.shape[0], dtype=torch.bool, device=src_ids.device)
    for _ in range(max_new_tokens):
        next_id = model.decode(ids, memory, src_mask)[:, -1].argmax(dim=-1)  # last row of Z only
        next_id = torch.where(finished, model.pad_id, next_id)
        ids = torch.cat([ids, next_id[:, None]], dim=1)
        finished |= next_id == eos_id
        if finished.all():
            break
    return ids

先手动走两步，看前缀怎么变长：

In [38]:
model.eval()
with torch.no_grad():
    memory, memory_mask = model.encode(s), model.source_mask(s)    # the encoder runs once
    ids = torch.tensor([[BOS]])
    for _ in range(2):
        Z_prefix = model.decode(ids, memory, memory_mask)          # [1, len(prefix), V]
        next_id = Z_prefix[:, -1].argmax(dim=-1)                   # only the last row matters
        print(f"prefix {list(ids.shape)} -> Z {list(Z_prefix.shape)} -> next word = {tgt_vocab[next_id.item()]}")
        ids = torch.cat([ids, next_id[:, None]], dim=1)
print("after 2 steps:", words(ids[0], tgt_vocab))

prefix [1, 1] -> Z [1, 1, 10] -> next word = <eos>
prefix [1, 2] -> Z [1, 2, 10] -> next word = tu
after 2 steps: <bos> <eos> tu


上面的手动循环不检查 EOS，所以 `<eos>` 之后还在继续。`greedy_decode` 会在 EOS 处停下。模型只更新过一次，输出没有意义：这里它第一步就生成了 `<eos>`，于是立刻结束。

In [39]:
ids = greedy_decode(model, s, max_new_tokens=6)
print(ids[0].tolist(), "=", words(ids[0], tgt_vocab))

[1, 2] = <bos> <eos>


### 额外：在这一句上训练几百步

这一块 `.py` 里没有。只有一个句对时，模型很快就能把它背下来；训练之后，贪心生成应该正好输出 `nous regardons video <eos>`。它把第 10 步（训练）和第 11 步（生成）连起来看。

In [40]:
torch.manual_seed(0)
toy = Transformer(len(src_vocab), len(tgt_vocab), d_model=d_model, num_heads=num_heads, d_ff=d_ff)
toy_optimizer = torch.optim.Adam(toy.parameters(), lr=1e-2)
for step in range(201):
    toy_loss = F.cross_entropy(toy(s, t).reshape(-1, len(tgt_vocab)), y.reshape(-1), ignore_index=PAD)
    toy_optimizer.zero_grad()
    toy_loss.backward()
    toy_optimizer.step()
    if step % 50 == 0:
        print(f"step {step:3d}   loss {toy_loss.item():.4f}")

toy.eval()
ids = greedy_decode(toy, s, max_new_tokens=6)
print("greedy decode:", words(ids[0], tgt_vocab))

step   0   loss 2.5236
step  50   loss 0.0401
step 100   loss 0.0098


step 150   loss 0.0052
step 200   loss 0.0033
greedy decode: <bos> nous regardons video <eos>


## RoPE：旋转位置编码

第 1 步在输入处把位置向量**加**到 embedding 上，只加一次。现在大多数 LLM 改用 RoPE（[RoFormer](https://arxiv.org/abs/2104.09864)）：第 1 步不加任何东西；在每一层的自注意力里，分头之后、打分之前，把 $Q$ 和 $K$ 的每一行按它所在的位置**旋转**一个角度。$V$ 不动。在编码器–解码器里，这指第 2 步和第 6 步；第 7 步的交叉注意力通常不转，因为它的 query 和 key 来自两个不同的序列，位置差没有意义。

| | 本笔记本 | 用 RoPE |
|---|---|---|
| 第 1 步 | $X = \sqrt{d}\, E[\mathbf{s}] + P$ | $X = E[\mathbf{s}]$，不加位置 |
| 第 2 步的投影 | $Q_i = X W_i^{Q}$，$K_i = X W_i^{K}$，$V_i = X W_i^{V}$ | 相同 |
| 第 2 步的打分 | $Q_i K_i^{\top} / \sqrt{d_k}$ | 先旋转 $Q_i$ 和 $K_i$ 的每一行，再算同样的缩放点积 |

把 $d_k$ 个特征两两配对 $(x_{2j}, x_{2j+1})$，每一对看成平面上的一个点。位置 $p$ 的第 $j$ 对转过角度 $p\,\theta_j$：

$$
\begin{pmatrix} x'_{2j} \\ x'_{2j+1} \end{pmatrix} = \begin{pmatrix} \cos p\theta_j & -\sin p\theta_j \\ \sin p\theta_j & \cos p\theta_j \end{pmatrix} \begin{pmatrix} x_{2j} \\ x_{2j+1} \end{pmatrix}
$$

把这个 2×2 旋转矩阵记作 $R(p\theta_j)$，$\theta_j$ 是第 $j$ 对的转速（见下面的 `speeds`）；不带下标的 $R(p\theta)$ 表示把每一对 $j$ 各转 $p\theta_j$ 的分块对角矩阵。关键性质是 $R(a)^{\top} R(b) = R(b - a)$：先转 $b$ 再倒转 $a$，等于只转 $b - a$。所以位置 $p$ 的 query 和位置 $p'$ 的 key 的点积只取决于相对距离 $p' - p$：

$$
\big(R(p\theta)\, q\big) \cdot \big(R(p'\theta)\, k\big) = q \cdot R\big((p' - p)\,\theta\big)\, k
$$

第 1 步后面的思考题已经用具体数字验证过这一点。

### `speeds`：每一对的转速

$\theta_j = \text{base}^{-j/m}$，$j = 0, \dots, m-1$。第 0 对每个位置转 1 弧度，后面的对越来越慢。这里的 $m$ 是速度表的长度，和前面的目标句长度 $m$ 无关：

- **文本**：$m = d_k/2$，每一对一个速度。
- **图像**：$m = d_k/4$，前一半的对随行号转、后一半随列号转，两半用同一张表。
- **视频**：$m = d_k/2$，一张从快到慢的表，按 $j \bmod 3$ 轮流分给 $t, h, w$。

In [41]:
def speeds(m: int, base: float = 10000.0) -> Tensor:
    """theta_0 .. theta_{m-1} = base^(-j/m): from 1 radian per position down to almost 0."""
    return base ** (-torch.arange(m, dtype=torch.float64) / m)

In [42]:
print("theta for m = 4:", speeds(4).tolist())                      # 1, 0.1, 0.01, 0.001

theta for m = 4: [1.0, 0.1, 0.01, 0.001]


### `angles_1d`：文本

位置 $p$ 的 token，第 $j$ 对转 $p\,\theta_j$。**输入** 位置 `[L]`，**输出** 角度 `[L, d_k/2]`。速度 $\theta_j = 10000^{-2j/d_k}$ 和第 1 步的 $\omega_i = 10000^{-2i/d}$ 是同一个公式，只是把 $d$ 换成每个头的宽度 $d_k$。这里 $d_k = 4$，只有两个速度 $1$ 和 $0.01$；第 1 步 $d = 8$ 时是 $1, 0.1, 0.01, 0.001$。另一个区别是角度不再写进 $X$，而是用来旋转 $Q$ 和 $K$。

In [43]:
def angles_1d(p: Tensor, d_k: int) -> Tensor:
    """Text. Token at position p: every pair j turns by p * theta_j.  [L] -> [L, d_k/2]"""
    return p[:, None] * speeds(d_k // 2)

In [44]:
angle_text = angles_1d(torch.arange(5), d_k=4)                     # [5, 2]: pair 0 turns by p * 1, pair 1 by p * 0.01
print(angle_text)

tensor([[0.00, 0.00],
        [1.00, 0.01],
        [2.00, 0.02],
        [3.00, 0.03],
        [4.00, 0.04]], dtype=torch.float64)


### `rotate`：把每一对转一个角度

偶数下标的特征 $x_{2j}$ 和奇数下标的特征 $x_{2j+1}$ 分别取出来，套用上面的 2×2 旋转，再用 `stack(..., dim=-1).flatten(-2)` 交错拼回去。**输入** `x: [..., L, d_k]` 和 `angle: [L, d_k/2]`，**输出** 形状与 `x` 相同。

In [45]:
def rotate(x: Tensor, angle: Tensor) -> Tensor:
    """Turn pair j of row p of x by angle[p, j].  x: [..., L, d_k], angle: [L, d_k/2].

    In attention this is applied to the rows of Q and K after the head split and before the
    scores; V is left alone.
    """
    cos, sin = angle.cos().to(x), angle.sin().to(x)
    even, odd = x[..., 0::2], x[..., 1::2]            # x_2j and x_2j+1
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1).flatten(-2)

旋转不改变长度；位置 0 的角度是 0，什么都不变；位置 1 的第 0 对正好被转了 1 弧度：

In [46]:
torch.manual_seed(8)
x = torch.randn(5, 4)                                              # 5 positions, d_k = 4
x_rotated = rotate(x, angle_text)
print("norms before:", x.norm(dim=-1))
print("norms after: ", x_rotated.norm(dim=-1))
assert torch.allclose(x_rotated[0], x[0])                          # position 0 turns by 0
c1, s1 = math.cos(1.0), math.sin(1.0)
assert torch.allclose(x_rotated[1, :2], torch.stack([c1 * x[1, 0] - s1 * x[1, 1], s1 * x[1, 0] + c1 * x[1, 1]]))
print("pair 0 of position 1 is turned by exactly 1 radian")

norms before: tensor([2.81, 1.39, 2.13, 1.37, 0.98])
norms after:  tensor([2.81, 1.39, 2.13, 1.37, 0.98])
pair 0 of position 1 is turned by exactly 1 radian


### `rotary_scores`：带 RoPE 的第 2 步打分

先转 $q$ 和 $k$，再做缩放点积。验证相对位置性质：同一对向量放在位置 $(2, 5)$ 和放在位置 $(10, 13)$，距离都是 3，分数相同；距离变了，分数也变。

In [47]:
def rotary_scores(q: Tensor, k: Tensor, q_angle: Tensor, k_angle: Tensor) -> Tensor:
    """Scores of step 2 with RoPE: q [..., Lq, d_k], k [..., Lk, d_k] -> [..., Lq, Lk]."""
    return rotate(q, q_angle) @ rotate(k, k_angle).transpose(-2, -1) / math.sqrt(q.shape[-1])

In [48]:
torch.manual_seed(9)
q_vec, k_vec = torch.randn(1, 8), torch.randn(1, 8)                # one query and one key, d_k = 8


def score_at(p_query, p_key):
    q_angle = angles_1d(torch.tensor([p_query]), 8)
    k_angle = angles_1d(torch.tensor([p_key]), 8)
    return rotary_scores(q_vec, k_vec, q_angle, k_angle).item()


print(f"score(2, 5)   = {score_at(2, 5):.4f}")
print(f"score(10, 13) = {score_at(10, 13):.4f}   same distance, same score")
print(f"score(2, 6)   = {score_at(2, 6):.4f}   different distance")
assert math.isclose(score_at(2, 5), score_at(10, 13), abs_tol=1e-5)

score(2, 5)   = -1.1359
score(10, 13) = -1.1359   same distance, same score
score(2, 6)   = -1.3338   different distance


**放进注意力。** 按表格的写法，$X$ 只用 embedding，不加 $P$。用第 2 步 `self_attention` 的投影得到已经分好头的 $Q$、$K$（`[B, h, n, d_k]`），按位置 0–4 旋转后打分。分数的形状和第 2 步一样是 `[B, h, n, n]`，后面的 mask、softmax、乘 $V$ 都不变。

In [49]:
X_rope = src_embedding.embedding(s)                                # [B, n, d]: no P is added
q_rope = self_attention.w_q(X_rope).view(B, Lq, h, d_k).transpose(1, 2)   # [B, h, n, d_k]
k_rope = self_attention.w_k(X_rope).view(B, Lk, h, d_k).transpose(1, 2)
angle_seq = angles_1d(torch.arange(Lq), d_k)                       # [n, d_k/2]
rope_scores = rotary_scores(q_rope, k_rope, angle_seq, angle_seq)  # [B, h, n, n]
print("RoPE scores", list(rope_scores.shape))

RoPE scores [1, 2, 5, 5]


### `angles_2d` 和 `angles_3d`：图像和视频

- **图像。** 第 $h$ 行、第 $w$ 列的 patch：前一半的对随 $h$ 转，后一半随 $w$ 转，每一半有 $d_k/4$ 个速度。两个 patch 的分数只取决于行距和列距。
- **视频。** 第 $t$ 帧、第 $h$ 行、第 $w$ 列的 patch：各对轮流分给 $t, h, w, t, h, w, \dots$，第 $j$ 对属于坐标 $j \bmod 3$，转速是 $\theta_j$。

In [50]:
def angles_2d(h: Tensor, w: Tensor, d_k: int) -> Tensor:
    """Image. Patch in row h, column w: half of the pairs turn with h, the other half with w."""
    theta = speeds(d_k // 4)
    return torch.cat([h[:, None] * theta, w[:, None] * theta], dim=1)

In [51]:
rows = torch.tensor([0, 0, 1, 1])                                  # a 2 x 2 grid of patches, read row by row
cols = torch.tensor([0, 1, 0, 1])
angle_image = angles_2d(rows, cols, d_k=8)                         # [4, 4]: 2 pairs follow h, 2 pairs follow w
print(angle_image)

tensor([[0.00, 0.00, 0.00, 0.00],
        [0.00, 0.00, 1.00, 0.01],
        [1.00, 0.01, 0.00, 0.00],
        [1.00, 0.01, 1.00, 0.01]], dtype=torch.float64)


同一行的 patch 0 和 1，前两个角度（随 $h$ 转的两对）相同；同一列的 patch 0 和 2，后两个角度（随 $w$ 转的两对）相同。

In [52]:
def angles_3d(t: Tensor, h: Tensor, w: Tensor, d_k: int) -> Tensor:
    """Video. Patch in frame t, row h, column w: the pairs are dealt out t, h, w, t, h, w, ..."""
    position = torch.stack([t, h, w], dim=1)          # [L, 3]
    owner = torch.arange(d_k // 2) % 3                # pair j belongs to coordinate j mod 3
    return position[:, owner] * speeds(d_k // 2)

In [53]:
frame, row, col = torch.tensor([0, 1]), torch.tensor([1, 1]), torch.tensor([2, 2])   # same (h, w), two frames
angle_video = angles_3d(frame, row, col, d_k=12)                   # [2, 6]: pairs 0..5 belong to t, h, w, t, h, w
print("owner of each pair:", ["thw"[j % 3] for j in range(6)])
print(angle_video)

owner of each pair: ['t', 'h', 'w', 't', 'h', 'w']
tensor([[    0.00,     0.22,     0.09,     0.00,     0.00,     0.00],
        [    1.00,     0.22,     0.09,     0.01,     0.00,     0.00]],
       dtype=torch.float64)


两个 patch 位置相同、帧不同，所以只有属于 $t$ 的第 0 对和第 3 对角度不同。

## 最后：一次跑完 `main()`

`.py` 的 `main()` 把上面的流程从头到尾串起来：重新设随机种子、建模型、按公式顺序逐步打印每个形状、检查逐步计算等于 `model(s, t)`、打印因果 mask 和一个头的注意力权重、做一次参数更新（第 10 步）、贪心生成（第 11 步）。它开头调用了 `torch.manual_seed(0)`，所以输出和书里印出的结果完全一致。

- **数字为什么和前面不同。** 前面逐步运行时，每个模块建之前各设了一个种子（`torch.manual_seed(0)` 到 `(7)`），这样单独重跑建某个模块的那一格，得到的权重不变。`main()` 在 `torch.manual_seed(0)` 之后一次建好整个模型：第一个建的源 embedding 表和第 1 步的相同，其余权重都不同，所以注意力权重、loss 和生成结果都和前面不一样，只有形状和各项检查可以直接对照。"额外"那一块训练循环打印的 `step 0` loss 也是 2.5236：那里同样在 `torch.manual_seed(0)` 之后新建了一个同样配置的 `Transformer`，初始权重和 `main()` 里的完全相同。
- **下标。** `main()` 打印的 `t_1..t_j` 沿用 `.py` 从 1 数的写法（`t_1` 是 BOS），和本笔记本的 $t_0, \dots, t_j$ 一样，都表示从 BOS 到当前位置的前缀；同一个位置，从 1 数的下标比这里大 1。

In [54]:
def main() -> None:
    torch.manual_seed(0)
    torch.set_printoptions(precision=2, sci_mode=False)
    src_vocab = ["<pad>", "<bos>", "<eos>", "we", "watch", "a", "short", "video"]
    tgt_vocab = ["<pad>", "<bos>", "<eos>", "nous", "regardons", "video", "une", "courte", "tu", "regardes"]
    s = torch.tensor([[3, 4, 5, 6, 7]])   # we watch a short video     [B=1, n=5]
    t = torch.tensor([[1, 3, 4, 5]])      # BOS nous regardons video   [B=1, m=4]
    y = torch.tensor([[3, 4, 5, 2]])      # nous regardons video EOS   [B=1, m=4]
    print("s =", s[0].tolist(), "=", " ".join(src_vocab[i] for i in s[0]))
    print("t =", t[0].tolist(), "=", " ".join(tgt_vocab[i] for i in t[0]))
    print("y =", y[0].tolist(), "=", " ".join(tgt_vocab[i] for i in y[0]), "\n")

    model = Transformer(len(src_vocab), len(tgt_vocab), d_model=8, num_heads=2, d_ff=32)
    enc, dec = model.encoder[0], model.decoder[0]
    src_mask, tgt_mask = model.source_mask(s), model.target_mask(t)

    def show(step: str, name: str, tensor: Tensor) -> Tensor:
        print(f"step {step:<3} {name:<38} {list(tensor.shape)}")
        return tensor

    X = show("1", "X = sqrt(d) E_src[s] + P", model.src_embedding(s))
    A = show("2", "A = MHA(X, X, X)", enc.self_attention(X, X, X, src_mask))
    show("2", "    softmax weights, per head", enc.self_attention.attention_weights)
    X1 = show("3", "X1 = LN(X + A)", enc.norm1(X + A))
    H = show("4", "H = LN(X1 + FFN(X1))", enc.norm2(X1 + enc.ffn(X1)))
    Y = show("5", "Y = sqrt(d) E_tgt[t] + P", model.tgt_embedding(t))
    Y1 = show("6", "Y1 = LN(Y + MHA(Y, Y, Y; causal))", dec.norm1(Y + dec.self_attention(Y, Y, Y, tgt_mask)))
    show("6", "    causal weights, per head", dec.self_attention.attention_weights)
    Y2 = show("7", "Y2 = LN(Y1 + MHA(Y1, H, H; pad))", dec.norm2(Y1 + dec.cross_attention(Y1, H, H, src_mask)))
    show("7", "    cross weights, per head", dec.cross_attention.attention_weights)
    Y3 = show("8", "Y3 = LN(Y2 + FFN(Y2))", dec.norm3(Y2 + dec.ffn(Y2)))
    Z = show("9", "Z = Y3 W_vocab + b", model.vocab_projection(Y3))
    assert torch.allclose(Z, model(s, t))  # the step-by-step path is exactly model.forward

    print("\ncausal mask, rows = queries, columns = keys (1 = may read):")
    print(tgt_mask[0, 0].int())
    print("\ndecoder self-attention weights, head 0 (rows sum to 1; zeros above the diagonal):")
    print(dec.self_attention.attention_weights[0, 0].detach())

    loss = F.cross_entropy(Z.reshape(-1, len(tgt_vocab)), y.reshape(-1), ignore_index=model.pad_id)
    print(f"\nstep 10  loss = -mean_j log p(y_j | t_1..t_j, s) = {loss.item():.4f}")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss.backward()    # dL/dtheta for every parameter, encoder included
    optimizer.step()   # one update; a real model needs many thousands

    model.eval()
    ids = greedy_decode(model, s, max_new_tokens=6)
    print("step 11  greedy decode:", ids[0].tolist(), "=",
          " ".join(tgt_vocab[i] for i in ids[0]), "(untrained, so the words are arbitrary)")

In [55]:
main()

s = [3, 4, 5, 6, 7] = we watch a short video
t = [1, 3, 4, 5] = <bos> nous regardons video
y = [3, 4, 5, 2] = nous regardons video <eos> 

step 1   X = sqrt(d) E_src[s] + P               [1, 5, 8]
step 2   A = MHA(X, X, X)                       [1, 5, 8]
step 2       softmax weights, per head          [1, 2, 5, 5]
step 3   X1 = LN(X + A)                         [1, 5, 8]
step 4   H = LN(X1 + FFN(X1))                   [1, 5, 8]
step 5   Y = sqrt(d) E_tgt[t] + P               [1, 4, 8]
step 6   Y1 = LN(Y + MHA(Y, Y, Y; causal))      [1, 4, 8]
step 6       causal weights, per head           [1, 2, 4, 4]
step 7   Y2 = LN(Y1 + MHA(Y1, H, H; pad))       [1, 4, 8]
step 7       cross weights, per head            [1, 2, 4, 5]
step 8   Y3 = LN(Y2 + FFN(Y2))                  [1, 4, 8]
step 9   Z = Y3 W_vocab + b                     [1, 4, 10]

causal mask, rows = queries, columns = keys (1 = may read):
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype

## 形状总表

$n = 5$，$m = 4$，$d = 8$，$h = 2$，$d_k = 4$，$d_{ff} = 32$，$V_{\text{tgt}} = 10$。开头的 `1` 是 batch 维 $B$。

| 步骤 | 输入 | 输出 |
|---|---|---|
| 1 · 源 embedding + 位置 | $\mathbf{s}$ `[1, 5]` | $X$ `[1, 5, 8]` |
| 2 · 每头的 $Q_i, K_i, V_i$ | $X$ `[1, 5, 8]` | 各 `[1, 2, 5, 4]` |
| 2 · 分数 $Q_i K_i^{\top} / \sqrt{d_k}$ | `[1, 2, 5, 4]` 和 `[1, 2, 4, 5]` | `[1, 2, 5, 5]` |
| 2 · 权重乘 $V_i$ | `[1, 2, 5, 5]` 和 `[1, 2, 5, 4]` | `[1, 2, 5, 4]` |
| 2 · 拼接各头，乘 $W^{O}$ | `[1, 5, 8]` | $A$ `[1, 5, 8]` |
| 3 · Add & Norm | $X, A$ `[1, 5, 8]` | $X_1$ `[1, 5, 8]` |
| 4 · FFN + Add & Norm | $X_1$ `[1, 5, 8]`，中间 `[1, 5, 32]` | $H$ `[1, 5, 8]` |
| 5 · 目标 embedding + 位置 | $\mathbf{t}$ `[1, 4]` | $Y$ `[1, 4, 8]` |
| 6 · 掩码自注意力（分数 `[1, 2, 4, 4]`）+ Add & Norm | $Y$ `[1, 4, 8]` | $Y_1$ `[1, 4, 8]` |
| 7 · 交叉注意力（分数 `[1, 2, 4, 5]`）+ Add & Norm | $Y_1$ `[1, 4, 8]`，$H$ `[1, 5, 8]` | $Y_2$ `[1, 4, 8]` |
| 8 · FFN + Add & Norm | $Y_2$ `[1, 4, 8]`，中间 `[1, 4, 32]` | $Y_3$ `[1, 4, 8]` |
| 9 · 词表投影 | $Y_3$ `[1, 4, 8]` | $Z$ `[1, 4, 10]` |
| 10 · 交叉熵 | $Z$ `[1, 4, 10]`，$\mathbf{y}$ `[1, 4]` | $\mathcal{L}$ 标量 |
| 11 · 贪心生成 | $\mathbf{s}$ `[1, 5]` | 至多 `1 + max_new_tokens` 个 id |

**参数量和计算量。** 一个编码器层有 $4d^2$ 个注意力权重（所有头的 $W^{Q}, W^{K}, W^{V}, W^{O}$）和 $2\,d\,d_{ff}$ 个 FFN 权重，另有少量偏置和 LayerNorm 参数。解码器层有两个注意力块，共 $8d^2 + 2\,d\,d_{ff}$。每层的计算量是这些权重乘以 token 数，再加上注意力分数的 $O(h\,n^2 d_k) = O(n^2 d)$，这是唯一随序列长度平方增长的项。